# Pakistani Fashion Intelligence
## Step 1 — Web Scraping and Raw Data Collection

**Project:** Fashion Brand Competitive Price Analytics  
**Prepared by:** Aqib Hanif  
**Assigned By:** Mam Sumayyea Salahuddin  
**Institute:** Arfa Karim Incubation Center, Peshawar  

### About this notebook

In this notebook, I collect product information from the official websites of three Pakistani fashion brands:

- J. (Junaid Jamshed)
- Maria.B
- Sana Safinaz

My purpose is to create the raw dataset for my fashion competitive intelligence project. I will use this data later for cleaning, PostgreSQL analysis, and the final interactive Excel dashboard.

I am focusing on these main categories:

- Unstitched
- Pret / Ready to Wear
- Stitched
- Luxury Pret
- Formal

I am aiming for around **100 products per brand**, so the complete dataset can contain approximately **300 products**.

## 1. Import the required libraries

I use:

- **pandas** to store the collected data in table form.
- **requests** to open the brand websites.
- **BeautifulSoup** to read product links from web pages.
- **json and re** to extract and clean product information.
- **time** to add a small delay between requests.
- **datetime** to save the scraping date.

In [1]:
# I import the libraries that I need for web scraping and data handling.

import json
import re
import time
from datetime import date
from math import ceil
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Project settings

Here I keep the main settings in one place. This makes the notebook easier for me to update later.

I selected **100 products per brand** as my target. I also use a small request delay so I do not send too many requests to a website at once.

In [2]:
# I keep my main scraping settings here so I can change them easily.

TARGET_PER_BRAND = 100
REQUEST_DELAY = 0.7
MAX_COLLECTION_PAGES = 6

# This will be the first raw data file of my project.
OUTPUT_FILE = Path("01_raw_fashion_products.csv")

# I use a normal browser user-agent because some websites reject blank/default requests.
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

print("Target products:", TARGET_PER_BRAND * 3)

Target products: 300


## 3. Brand collection pages

I selected collection pages from the **official websites** of the three brands.

Using different categories is important because I do not want my dataset to represent only one type of clothing.

In [3]:
# I store the official collection pages category-wise.
# This also helps me label each product with the correct main category.

COLLECTIONS = {
    "J.": [
        ("Unstitched", "Women Unstitched",
         "https://www.junaidjamshed.com/collections/womens-new-in-women-unstitched"),
        ("Pret", "Women Ready to Wear",
         "https://www.junaidjamshed.com/collections/womens-new-in-women-ready-to-wear"),
        ("Formal", "Women Formals",
         "https://www.junaidjamshed.com/collections/womens-formals"),
    ],

    "Maria.B": [
        ("Unstitched", "Unstitched",
         "https://www.mariab.pk/collections/unstitched-fabrics"),
        ("Stitched", "Stitched",
         "https://www.mariab.pk/collections/stitched"),
        ("Luxury Pret", "Luxury Pret",
         "https://www.mariab.pk/collections/luxury-pret"),
        ("Formal", "Luxury Formals",
         "https://www.mariab.pk/collections/luxury-formals"),
    ],

    "Sana Safinaz": [
        ("Unstitched", "Unstitched Fabric",
         "https://sanasafinaz.com/collections/unstitched-fabric"),
        ("Pret", "Ready to Wear",
         "https://sanasafinaz.com/collections/ready-to-wear-2"),
        ("Luxury Pret", "Luxury Pret",
         "https://sanasafinaz.com/collections/luxury-pret"),
        ("Formal", "Formals",
         "https://sanasafinaz.com/collections/formals"),
    ],
}

print("Brands selected:", list(COLLECTIONS.keys()))

Brands selected: ['J.', 'Maria.B', 'Sana Safinaz']


## 4. Create a web session

I use one `requests` session for the scraping process. This keeps my request settings consistent for all three websites.

In [4]:
# I create one session and apply my browser headers to it.

session = requests.Session()
session.headers.update(HEADERS)

print("Web session is ready.")

Web session is ready.


## 5. Helper functions

Before scraping products, I create a few small helper functions.

Their purpose is to:

- clean extra spaces from text,
- convert price text into numbers,
- identify a product from its URL,
- and safely open a webpage.

In [5]:
# I use this function to remove extra spaces from product names and other text.
def clean_text(value):
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


# I use this function to convert values such as "Rs. 12,990" into 12990.
def money_to_float(value):
    if value is None or value == "":
        return None

    if isinstance(value, (int, float)):
        return float(value)

    text = str(value).replace(",", "")
    match = re.search(r"\d+(?:\.\d+)?", text)

    return float(match.group()) if match else None


# Shopify can store a price in the smallest currency unit.
# I convert it into a normal price for my dataset.
def shopify_money(value):
    if value is None or value == "":
        return None

    try:
        return float(value) / 100
    except (TypeError, ValueError):
        return money_to_float(value)


# I get the unique product handle from the end of a product URL.
def product_handle(product_url):
    path = urlparse(product_url).path.rstrip("/")
    return path.split("/")[-1]


# I use this function whenever I need to open a webpage.
# If the page cannot be opened, my notebook continues instead of stopping.
def get_html(url):
    try:
        response = session.get(url, timeout=25)
        response.raise_for_status()
        return response.text

    except requests.RequestException as error:
        print(f"Could not open: {url}")
        print("Reason:", error)
        return None

## 6. Collect product links

Collection pages normally contain links to many individual products.

This function goes through the collection pages and collects unique product URLs. I stop when a page gives no new product links.

In [6]:
# I collect individual product URLs from a category/collection page.
def collect_product_links(collection_url, page_limit=MAX_COLLECTION_PAGES):

    product_links = []
    seen = set()

    for page in range(1, page_limit + 1):

        # I add the page number so I can collect products from more than one page.
        separator = "&" if "?" in collection_url else "?"
        page_url = f"{collection_url}{separator}page={page}"

        html = get_html(page_url)

        if not html:
            break

        soup = BeautifulSoup(html, "html.parser")
        new_links = 0

        # Most Shopify fashion websites use /products/ in product URLs.
        for anchor in soup.select('a[href*="/products/"]'):

            href = anchor.get("href")

            if not href:
                continue

            full_url = urljoin(collection_url, href.split("?")[0])
            handle = product_handle(full_url)

            if handle and handle not in seen:
                seen.add(handle)
                product_links.append(full_url)
                new_links += 1

        # I stop if this page does not contain any new products.
        if new_links == 0:
            break

        time.sleep(REQUEST_DELAY)

    return product_links

## 7. Read product information

I first try Shopify's product JSON endpoint because it usually gives cleaner product information than reading visible HTML.

If that method does not work, I use structured product data (`JSON-LD`) from the product page as a fallback.

In [7]:
# I search structured JSON-LD data for a Product object.
# This is my backup method if the cleaner Shopify endpoint is not available.
def find_product_jsonld(html):

    soup = BeautifulSoup(html, "html.parser")

    def search_object(obj):

        if isinstance(obj, dict):

            obj_type = obj.get("@type")

            if obj_type == "Product" or (
                isinstance(obj_type, list) and "Product" in obj_type
            ):
                return obj

            for value in obj.values():
                result = search_object(value)

                if result:
                    return result

        elif isinstance(obj, list):

            for value in obj:
                result = search_object(value)

                if result:
                    return result

        return None

    for script in soup.find_all("script", type="application/ld+json"):

        try:
            data = json.loads(script.string or script.get_text())
        except (json.JSONDecodeError, TypeError):
            continue

        product = search_object(data)

        if product:
            return product

    return None

In [8]:
# I try to collect the main product details from Shopify's product JSON.
def parse_shopify_product(product_url):

    js_url = product_url.rstrip("/") + ".js"

    try:
        response = session.get(js_url, timeout=25)

        if response.ok:

            data = response.json()
            variants = data.get("variants", [])

            if not variants:
                return None

            # I collect all current prices from the product variants.
            current_prices = [
                shopify_money(v.get("price"))
                for v in variants
            ]
            current_prices = [
                price for price in current_prices
                if price is not None
            ]

            # I also collect compare-at prices to identify discounts.
            compare_prices = [
                shopify_money(v.get("compare_at_price"))
                for v in variants
            ]
            compare_prices = [
                price for price in compare_prices
                if price is not None and price > 0
            ]

            sale_price = min(current_prices) if current_prices else None
            original_price = (
                max(compare_prices)
                if compare_prices
                else sale_price
            )

            # I keep one available SKU if it exists.
            sku = next(
                (
                    clean_text(v.get("sku"))
                    for v in variants
                    if v.get("sku")
                ),
                ""
            )

            # If at least one variant is available, I mark the product as available.
            availability = (
                "Available"
                if any(bool(v.get("available")) for v in variants)
                else "Out of Stock"
            )

            return {
                "Product_Name": clean_text(data.get("title")),
                "SKU": sku,
                "Product_Type": clean_text(data.get("type")),
                "Original_Price": original_price,
                "Sale_Price": sale_price,
                "Availability": availability,
            }

    except (requests.RequestException, ValueError):
        pass

    return None

In [9]:
# I use this backup parser when the Shopify product JSON does not work.
def parse_product_fallback(product_url):

    html = get_html(product_url)

    if not html:
        return None

    product = find_product_jsonld(html)

    if not product:
        return None

    offers = product.get("offers", {})

    if isinstance(offers, list):
        offers = offers[0] if offers else {}

    if offers.get("@type") == "AggregateOffer":
        sale_price = money_to_float(offers.get("lowPrice"))
        availability = "Available"

    else:
        sale_price = money_to_float(offers.get("price"))

        availability_text = clean_text(
            offers.get("availability")
        ).lower()

        availability = (
            "Out of Stock"
            if "outofstock" in availability_text
            else "Available"
        )

    return {
        "Product_Name": clean_text(product.get("name")),
        "SKU": clean_text(product.get("sku")),
        "Product_Type": "",
        "Original_Price": sale_price,
        "Sale_Price": sale_price,
        "Availability": availability,
    }

## 8. Create a simple subcategory

The websites do not always use the same subcategory names. For my first raw dataset, I use a small rule based on the product name.

I can improve these categories later during the data-cleaning step.

In [10]:
# I create a simple subcategory from words found in the product name.
def guess_subcategory(product_name):

    name = product_name.lower()

    if "3 piece" in name or "3pc" in name or "3-pc" in name:
        return "3 Piece"

    if "2 piece" in name or "2pc" in name or "2-pc" in name:
        return "2 Piece"

    if "shirt" in name and "trouser" in name:
        return "Shirt + Trouser"

    if "shirt" in name and "dupatta" in name:
        return "Shirt + Dupatta"

    if "saree" in name or "sari" in name:
        return "Saree"

    if "kurta" in name or "kurti" in name:
        return "Kurta"

    return "Other" 

## 9. Main scraping function

Now I combine all the previous functions.

For each brand, I:

1. open the selected collection pages,
2. collect individual product URLs,
3. read product details,
4. calculate discount amount and discount percentage,
5. and add the result to my raw dataset.

I try to keep the sample reasonably balanced across the selected categories.

In [11]:
# This is my main function for collecting the complete raw dataset.
def scrape_project_data():

    rows = []
    snapshot = date.today().isoformat()

    for brand, collections in COLLECTIONS.items():

        print(f"\n--- {brand} ---")

        brand_rows = 0
        seen_products = set()

        # I divide my target between the available categories of each brand.
        per_collection_target = ceil(
            TARGET_PER_BRAND / len(collections)
        )

        for category, collection_name, collection_url in collections:

            if brand_rows >= TARGET_PER_BRAND:
                break

            print("Collecting:", category)

            links = collect_product_links(collection_url)
            collected_here = 0

            for product_url in links:

                if (
                    brand_rows >= TARGET_PER_BRAND
                    or collected_here >= per_collection_target
                ):
                    break

                handle = product_handle(product_url)

                # I skip a product if I have already collected it.
                if handle in seen_products:
                    continue

                # I try the clean Shopify method first.
                product = parse_shopify_product(product_url)

                # If it fails, I use my backup parser.
                if product is None:
                    product = parse_product_fallback(product_url)

                # I keep only records with a product name and a usable price.
                if (
                    not product
                    or not product.get("Product_Name")
                    or product.get("Sale_Price") is None
                ):
                    continue

                original_price = (
                    product.get("Original_Price")
                    or product.get("Sale_Price")
                )

                sale_price = product.get("Sale_Price")

                # Original price should not be lower than the current sale price.
                if original_price < sale_price:
                    original_price = sale_price

                # I calculate the discount fields for later analysis.
                discount_amount = round(
                    original_price - sale_price,
                    2
                )

                discount_percent = (
                    round(
                        (discount_amount / original_price) * 100,
                        2
                    )
                    if original_price
                    else 0.0
                )

                rows.append({
                    "Product_ID": (
                        f"{brand.replace('.', '').replace(' ', '')[:3].upper()}"
                        f"-{brand_rows + 1:03d}"
                    ),
                    "Snapshot_Date": snapshot,
                    "Brand": brand,
                    "Product_Name": product["Product_Name"],
                    "SKU": product.get("SKU", ""),
                    "Category": category,
                    "Subcategory": guess_subcategory(
                        product["Product_Name"]
                    ),
                    "Collection": collection_name,
                    "Original_Price": round(original_price, 2),
                    "Sale_Price": round(sale_price, 2),
                    "Discount_Amount": discount_amount,
                    "Discount_Percent": discount_percent,
                    "Availability": product.get(
                        "Availability",
                        "Unknown"
                    ),
                    "Currency": "PKR",
                    "Product_URL": product_url,
                    "Source_Website": urlparse(
                        product_url
                    ).netloc,
                    "Scrape_Date": snapshot,
                })

                seen_products.add(handle)
                brand_rows += 1
                collected_here += 1

                time.sleep(REQUEST_DELAY)

        print(f"Collected {brand_rows} products for {brand}.")

    df = pd.DataFrame(rows)

    # I remove duplicate product links before saving the final raw dataset.
    if not df.empty:
        df = (
            df
            .drop_duplicates(
                subset=["Brand", "Product_URL"]
            )
            .reset_index(drop=True)
        )

    return df

## 10. Run the scraper

This is the main execution cell.

**Note:** Web scraping depends on internet access and website structure. If a brand changes its website layout, I may need to update that part of the scraper.

In [12]:
# I run my scraper and store the collected records in a DataFrame called raw_df.

raw_df = scrape_project_data()

print("\nScraping finished.")
print("Total products collected:", len(raw_df))


--- J. ---
Collecting: Unstitched
Collecting: Pret
Collecting: Formal
Collected 100 products for J..

--- Maria.B ---
Collecting: Unstitched
Collecting: Stitched
Collecting: Luxury Pret
Collecting: Formal
Collected 100 products for Maria.B.

--- Sana Safinaz ---
Collecting: Unstitched
Collecting: Pret
Collecting: Luxury Pret
Collecting: Formal
Collected 81 products for Sana Safinaz.

Scraping finished.
Total products collected: 281


## 11. Check the collected data

Before saving the file, I do a few simple checks.

I want to confirm:

- how many products I collected,
- how many came from each brand,
- which categories are present,
- and whether important columns contain missing values.

In [13]:
# I preview the first five rows to make sure the columns look correct.

raw_df.head()

,Product_ID,Snapshot_Date,Brand,Product_Name,SKU,Category,Subcategory,Collection,Original_Price,Sale_Price,Discount_Amount,Discount_Percent,Availability,Currency,Product_URL,Source_Website,Scrape_Date
0,J-001,2026-08-31,J.,Choco Vanille,PU300268-DFT-100ML-REG,Unstitched,Other,Women Unstitched,6900.0,6900.0,0.0,0.0,Available,PKR,https://www.junaidjamshed.com/products/choco-v...,www.junaidjamshed.com,2026-08-31
1,J-002,2026-08-31,J.,Galaxy Grape,PU300267-DFT-100ML-REG,Unstitched,Other,Women Unstitched,6900.0,6900.0,0.0,0.0,Available,PKR,https://www.junaidjamshed.com/products/galaxy-...,www.junaidjamshed.com,2026-08-31
2,J-003,2026-08-31,J.,Matcha Matcha,PU300266-DFT-100ML-REG,Unstitched,Other,Women Unstitched,6900.0,6900.0,0.0,0.0,Available,PKR,https://www.junaidjamshed.com/products/matcha-...,www.junaidjamshed.com,2026-08-31
3,J-004,2026-08-31,J.,Pink Sugar,PU300265-DFT-100ML-REG,Unstitched,Other,Women Unstitched,6900.0,6900.0,0.0,0.0,Available,PKR,https://www.junaidjamshed.com/products/pink-su...,www.junaidjamshed.com,2026-08-31
4,J-005,2026-08-31,J.,ENIGMA NOIR,PL300018-DFT-100ML-REG,Unstitched,Other,Women Unstitched,4500.0,4500.0,0.0,0.0,Available,PKR,https://www.junaidjamshed.com/products/enigma-...,www.junaidjamshed.com,2026-08-31


In [14]:
# I check how many products I collected from each brand.

if not raw_df.empty:
    display(
        raw_df["Brand"]
        .value_counts()
        .rename_axis("Brand")
        .reset_index(name="Products")
    )
else:
    print("No data is available to check.")

,Brand,Products
0,J.,100
1,Maria.B,100
2,Sana Safinaz,81


In [15]:
# I check the product count by brand and main category.

if not raw_df.empty:
    display(
        pd.crosstab(
            raw_df["Brand"],
            raw_df["Category"]
        )
    )
else:
    print("No data is available to check.")

Category,Formal,Luxury Pret,Pret,Stitched,Unstitched
Brand,,,,,
J.,32,0,34,0,34
Maria.B,25,25,0,25,25
Sana Safinaz,6,25,25,0,25


In [16]:
# I check missing values in the most important columns.

important_columns = [
    "Product_ID",
    "Brand",
    "Product_Name",
    "Category",
    "Original_Price",
    "Sale_Price",
    "Availability",
    "Product_URL",
]

if not raw_df.empty:
    display(
        raw_df[important_columns]
        .isna()
        .sum()
        .rename("Missing_Values")
        .to_frame()
    )
else:
    print("No data is available to check.")

,Missing_Values
Product_ID,0
Brand,0
Product_Name,0
Category,0
Original_Price,0
Sale_Price,0
Availability,0
Product_URL,0


## 12. Save the raw dataset

I save this file **before doing detailed cleaning** because I want to keep an original raw snapshot of the information collected from the websites.

This CSV will become the input for the next notebook step.

In [17]:
# I save my raw web-scraped data as a CSV file.

if not raw_df.empty:

    raw_df.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print("Raw dataset saved successfully.")
    print("File name:", OUTPUT_FILE)
    print("Rows:", len(raw_df))
    print("Columns:", len(raw_df.columns))

else:
    print("The file was not saved because no products were collected.")

Raw dataset saved successfully.
File name: 01_raw_fashion_products.csv
Rows: 281
Columns: 17


## 13. Step 1 conclusion

In this step, I prepared my web-scraping process for **J., Maria.B, and Sana Safinaz** and collected the basic product information required for my project.

The main output of this notebook is:

**`01_raw_fashion_products.csv`**

I will keep this file as my raw dataset.

### Next step

In the next step, I will clean and transform this raw data in Python. I will create fields such as:

- Price Segment
- Sale Status
- Discount Band
- Availability Score
- Brand Intelligence Score

These fields will later be used in PostgreSQL and in my final Excel dashboard.